# SVD (Singular Value Decomposition)

> 기준 COLMAP commit: `c004ddf51ee90bb70cbc2bb94a314c7f932febbe`

## SVD란?

**특이값 분해(SVD)**는 임의의 $m \times n$ 행렬 $A$를 세 행렬의 곱으로 분해하는 것입니다:

$$A = U \Sigma V^T$$

직관적으로, 행렬 $A$의 변환을 "회전 → 스케일링 → 회전"으로 분해한 것입니다.

### 각 행렬의 특징

| 행렬 | 크기 | 특징 |
|------|------|------|
| $U$ | $m \times m$ | 직교행렬 ($U^T U = I$). 열벡터들은 $AA^T$의 고유벡터 |
| $\Sigma$ | $m \times n$ | 대각행렬. 대각 원소가 **특이값** $\sigma_i$ |
| $V$ | $n \times n$ | 직교행렬 ($V^T V = I$). 열벡터들은 $A^T A$의 고유벡터 |

**핵심 포인트**:
- **특이값 정렬**: $\Sigma$의 대각 원소는 항상 **내림차순**으로 정렬됩니다: $\sigma_1 \geq \sigma_2 \geq ... \geq \sigma_n \geq 0$
- **V의 열 순서**: $V$의 $i$번째 열은 $\sigma_i$에 대응합니다. 즉, **V의 마지막 열이 가장 작은 특이값에 대응**
- **Null space와 해의 관계**: $\sigma_n = 0$이면 $V$의 마지막 열은 $A\mathbf{x} = \mathbf{0}$의 **정확한 해**입니다

### 왜 이게 중요한가?

우리가 풀고 싶은 문제는 $A\mathbf{x} = \mathbf{0}$입니다.

**Null space (영공간)**는 바로 이 방정식을 만족하는 모든 벡터 $\mathbf{x}$의 집합입니다.

따라서:
- $\sigma_n = 0$이면 → $V$의 마지막 열이 **정확한 해**
- $\sigma_n \approx 0$이면 → $V$의 마지막 열이 **최소 오차 근사해** (노이즈가 있는 실제 데이터)

## 왜 V의 마지막 열이 해인가?

COLMAP에서 SVD의 가장 중요한 용도는 **동차 선형 시스템의 해**를 구하는 것입니다:

$$A \mathbf{x} = \mathbf{0}$$

예를 들어 Triangulation, Homography, Fundamental Matrix 추정 등에서 이런 형태의 문제가 나타납니다.

> **요약**: $A\mathbf{x} = \mathbf{0}$의 해는 **가장 작은 특이값에 대응하는 V의 열**, 즉 **V의 마지막 열**입니다.

## COLMAP에서의 SVD 사용

COLMAP은 Eigen 라이브러리의 `JacobiSVD`를 사용합니다:

In [ ]:
Eigen::JacobiSVD<Eigen::MatrixXd> svd(A, Eigen::ComputeFullV);
solution = svd.matrixV().col(n-1);  // 마지막 열이 해



### 1. Triangulation (삼각측량)

두 카메라에서 관측된 2D 점으로부터 3D 점을 복원합니다.

**문제 설정**: 관측 방정식 $A \mathbf{X} = \mathbf{0}$을 구성

In [ ]:
// src/colmap/geometry/triangulation.cc
Eigen::Matrix4d A;
A.row(0) = cam_point1(0) * cam1_from_world.row(2) - cam1_from_world.row(0);
A.row(1) = cam_point1(1) * cam1_from_world.row(2) - cam1_from_world.row(1);
A.row(2) = cam_point2(0) * cam2_from_world.row(2) - cam2_from_world.row(0);
A.row(3) = cam_point2(1) * cam2_from_world.row(2) - cam2_from_world.row(1);

const Eigen::JacobiSVD<Eigen::Matrix4d> svd(A, Eigen::ComputeFullV);
*xyz = svd.matrixV().col(3).hnormalized();  // V의 4번째(마지막) 열



**코드 설명**:
- `cam1_from_world`: 카메라 1의 projection matrix $P_1$ (3×4)
- `cam_point1(0)`, `cam_point1(1)`: 이미지 상의 2D 점 좌표 $(u, v)$
- 각 행은 $u \cdot P_{row3} - P_{row1} = 0$ 형태의 제약조건 (DLT 방법)
- `matrixV().col(3)`: V의 4번째 열 = 동차좌표 $[X, Y, Z, W]^T$
- `.hnormalized()`: 동차좌표를 일반좌표로 변환 $[X/W, Y/Z, Z/W]^T$

### 2. Homography 추정

4개 이상의 점 대응으로부터 3×3 Homography 행렬을 추정합니다.

In [ ]:
// src/colmap/estimators/solvers/homography_matrix.cc
Eigen::JacobiSVD<Eigen::Matrix<double, Eigen::Dynamic, 9>> svd(
    A, Eigen::ComputeFullV);
const Eigen::VectorXd nullspace = svd.matrixV().col(8);  // V의 9번째(마지막) 열
H = Eigen::Map<const Eigen::Matrix3d>(nullspace.data()).transpose();



**코드 설명**:
- 3×3 Homography 행렬 $H$는 9개의 원소 → 벡터로 펼치면 9차원
- `matrixV().col(8)`: 9차원 벡터 (V의 9번째 = 마지막 열)
- `Eigen::Map<Matrix3d>(...).transpose()`: 9차원 벡터를 3×3 행렬로 재구성

### 3. Fundamental Matrix 추정

8개 이상의 점 대응으로부터 Fundamental Matrix를 추정합니다.

In [ ]:
// src/colmap/estimators/solvers/fundamental_matrix.cc
Eigen::JacobiSVD<Eigen::Matrix<double, Eigen::Dynamic, 9>> svd(
    A, Eigen::ComputeFullV);
Q = Eigen::Map<const Eigen::Matrix<double, 3, 3, Eigen::RowMajor>>(
    svd.matrixV().col(8).data());  // V의 9번째(마지막) 열

// rank-2 제약 적용 (F의 세 번째 특이값 = 0)
Eigen::JacobiSVD<Eigen::Matrix3d> svd2(
    Q, Eigen::ComputeFullU | Eigen::ComputeFullV);
Eigen::Vector3d singular_values = svd2.singularValues();
singular_values(2) = 0.0;  // 가장 작은 특이값을 0으로
F = svd2.matrixU() * singular_values.asDiagonal() * svd2.matrixV().transpose();



**코드 설명**:
- **1단계**: Homography와 동일하게 9차원 벡터를 3×3 행렬 $Q$로 변환
- **2단계 (rank-2 제약)**: Fundamental Matrix는 이론적으로 rank가 2여야 함
  - $Q$를 다시 SVD 분해: $Q = U_2 \Sigma_2 V_2^T$
  - 가장 작은 특이값 $\sigma_3$을 0으로 강제
  - $F = U_2 \cdot \text{diag}(\sigma_1, \sigma_2, 0) \cdot V_2^T$로 재구성
- 이렇게 하면 $\det(F) = 0$ 조건을 만족하는 유효한 Fundamental Matrix가 됨

## 정리

| 용도 | 행렬 크기 | 해 (V의 열) |
|------|----------|------------|
| Triangulation | 4×4 | `matrixV().col(3)` |
| Homography | N×9 | `matrixV().col(8)` |
| Fundamental Matrix | N×9 | `matrixV().col(8)` |

핵심 패턴:
1. 관측 방정식 $A\mathbf{x} = \mathbf{0}$ 구성
2. SVD 수행: `JacobiSVD<MatrixType> svd(A, ComputeFullV)`
3. **V의 마지막 열**이 해: `svd.matrixV().col(n-1)`

## 부록: 왜 V의 마지막 열이 해인가?

**목표**: $\|\mathbf{x}\| = 1$ 조건에서 $\|A\mathbf{x}\|$를 최소화하는 $\mathbf{x}$ 찾기

**Step 1**: SVD 대입

$$\|A\mathbf{x}\| = \|U\Sigma V^T \mathbf{x}\|$$

$U$는 직교(회전)행렬이므로 노름 불변:

$$= \|U\| \|\Sigma V^T \mathbf{x}\| = \|\Sigma V^T \mathbf{x}\|$$

**Step 2**: 변수 치환 $\mathbf{y} = V^T \mathbf{x}$

$V$도 직교행렬이므로 $\|\mathbf{y}\| = \|V^T \mathbf{x}\| = \|\mathbf{x}\| = 1$

$\Sigma$는 특이값들로 이루어진 대각행렬:

$$\Sigma = \begin{bmatrix} \sigma_1 & 0 & \cdots & 0 \\ 0 & \sigma_2 & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \cdots & \sigma_n \end{bmatrix}$$

따라서 $\Sigma \mathbf{y}$를 계산하면:

$$\|\Sigma \mathbf{y}\|^2 = \sigma_1^2 y_1^2 + \sigma_2^2 y_2^2 + ... + \sigma_n^2 y_n^2$$

**Step 3**: 최소화

목표: $\|\Sigma \mathbf{y}\|^2 = \sigma_1^2 y_1^2 + \sigma_2^2 y_2^2 + ... + \sigma_n^2 y_n^2$ 최소화

제약: $\|\mathbf{y}\|^2 = y_1^2 + y_2^2 + ... + y_n^2 = 1$

이것은 **가중 합(weighted sum)**입니다. 각 $y_i^2$에 가중치 $\sigma_i^2$가 곱해져 있습니다.

특이값 순서: $\sigma_1^2 \geq \sigma_2^2 \geq ... \geq \sigma_n^2$ (가중치가 점점 작아짐)

**핵심**: $y_1^2 + y_2^2 + ... + y_n^2 = 1$이라는 "예산"을 어디에 배분할지 선택하는 문제

→ **가장 작은 가중치** $\sigma_n^2$에 모든 예산을 몰아주면 전체 합이 최소!

따라서:
- $y_n = 1$ (가장 작은 가중치에 100%)
- 나머지 $y_i = 0$

$$\mathbf{y} = [0, 0, ..., 0, 1]^T$$

**Step 4**: 원래 변수로 복원

$$\mathbf{x} = V\mathbf{y} = V \cdot [0, 0, ..., 0, 1]^T = V\text{의 마지막 열}$$

## 참고 자료

[CMSC426 Computer Vision Math Tutorial](https://cmsc426.github.io/math-tutorial/#svd)

[SVD Visualized, Singular Value Decomposition explained | SEE Matrix , Chapter 3](https://www.youtube.com/watch?v=vSczTbgc8Rc)